# Discrete-Time Quantum Walk Cryptography

## Protocol: 
1. Alice samples at random l,s (initial state), k (coin operator), t(time) and evolves the initial state with a quantum walk ojn a ring of N sites, obtaining the public key
2. Bob encrypts the message by translating of m positions the public key
3. Alice performs the inverse qw and measures the state, recovering the message

## Quantum Walk Public-Key Cryptographic System — Implementation

Implementation of the protocol from Vlachou et al., *"Quantum walk public-key cryptographic system"* (2015), based on a discrete-time quantum walk on a ring of $N$ positions.

### Setup

The walker's state lives in $\mathcal H_p \otimes \mathcal H_c$, with position space $\mathcal H_p = \text{span}\{|i\rangle : i \in \{0,\dots,N-1\}\}$ and coin space $\mathcal H_c = \text{span}\{|L\rangle, |R\rangle\}$.

A single step of the walk is given by
$$\hat U_k = \hat S\,(\hat I_p \otimes \hat U_c(\theta_k,\xi_k,\zeta_k))$$

where $\hat S$ is the shift operator on the ring and $\hat U_c$ is the $2\times2$ coin operator, with $\theta_k=\xi_k=\zeta_k = k\cdot 2\pi/d$ for $k \in \mathcal I=\{1,\dots,d\}$ — a discretization of $U(2)$ into $d$ possible walks, used to keep the key space large enough for security.

### Public-key generation

Alice:
1. Chooses uniformly at random an initial position $l \in \{0,\dots,2^n-1\}$ and coin state $s \in \{L,R\}$;
2. Chooses uniformly at random a walk $\hat U_k$ ($k\in\mathcal I$) and a number of steps $t\in\mathcal T$;
3. Generates the public key by running the walk:
$$|\psi_{PK}\rangle = \hat U_k^{\,t}|l\rangle|s\rangle$$

implemented in `DTQW_RING(l, s, k, t)`.

### Secret key

$$SK = (\hat U_k, t, l)$$

### Message encryption

Bob obtains $|\psi_{PK}\rangle$, encodes the message $m$ as an integer, and encrypts it by applying a spatial translation:
$$|\psi(m)\rangle = (\hat T_m\otimes \hat I_c)\,|\psi_{PK}\rangle$$

implemented in `encrypt(message, private_key)` via `Translation_operator(m)`.

### Message decryption

Alice applies the inverse walk $\hat U_k^{-t} = (\hat U_k^{\,t})^\dagger$ to the received state. Since $\hat U_k^{\,t}$ commutes with $\hat T_m\otimes\hat I_c$ (Lemma 1 of the paper), this exactly undoes the walk and leaves:
$$|\psi_f\rangle = (\hat T_m\otimes\hat I_c)|l\rangle|s\rangle = |l+m \pmod N\rangle|s\rangle$$

Alice measures the position observable $\hat M = \sum_i |i\rangle\langle i|\otimes\hat I_c$. Since $|\psi_f\rangle$ is already an eigenstate of $\hat M$, the outcome $m' = l+m \pmod N$ is obtained with probability $1$ — the measurement is deterministic, not stochastic, by construction of the protocol. She recovers the message:
$$m = (m' - l) \pmod N$$

implemented in `decrypt(msg, l, s, k, t)`.

### Correctness constraint

$N \geq 2^n$, where $n$ is the number of bits of the message, is required so that distinct messages never collide modulo $N$ (Lemma 1's hypothesis).

### Security (not implemented, theoretical)

The mixed state $\hat\rho_{PK}$ perceived by an eavesdropper (averaged over $l,s$) is maximally mixed, $\hat\rho_{PK} = \frac{1}{2^{n+1}}\hat I_p\otimes\hat I_c$. By Holevo's theorem, the mutual information an eavesdropper can extract about $SK$ is bounded by $S(\hat\rho_{PK}) = n+1$ bits, which is exponentially smaller than the total entropy of the secret key $\log_2(d|\mathcal T|) + n+1$ when $d \approx \exp(n)$.

In [1]:
import numpy as np

max_letters = 1   # max letters of the msg
n = 8 * max_letters  # bits needed for the message (8 bits per char)

d    = 500   # number of quantum walks
Tmax = 10    # max time steps
N    = int(np.exp(n))   # ring size (N >= 2^n by protocol)
n_bits = n   # message length in bits — always a multiple of 8


def int_to_binary(m: int) -> str:
    # Converts an integer to a fixed n_bits binary string
    if m < 0 or m >= 2**n_bits:
        raise ValueError(f"m={m} not representable in {n_bits} bits (N={N})")
    return format(m, f'0{n_bits}b')


def binary_to_int(bits: str) -> int:
    # Converts a binary string to an integer
    return int(bits, 2)


def string_to_binary(message: str) -> str:
    # Converts a text string to a binary string (8 bits per char, UTF-8)
    return ''.join(format(byte, '08b') for byte in message.encode('utf-8'))


def binary_to_string(bits: str) -> str:
    # Converts a binary string (multiple of 8 bits) back to text
    pad = (8 - len(bits) % 8) % 8
    bits = '0' * pad + bits   # left-pad to nearest multiple of 8 if needed
    byte_chunks = [bits[i:i+8] for i in range(0, len(bits), 8)]
    return bytes(int(b, 2) for b in byte_chunks).decode('utf-8')


def int_to_string(m: int) -> str:
    # Converts an integer directly to a text string (via n_bits binary)
    return binary_to_string(int_to_binary(m))

In [2]:

def send_message_to_Alice(message):
    

    #sample initial state
    l=np.random.randint(0,N)
    s=np.random.randint(0,2)#coin
    k=np.random.randint(0,d)
    t=np.random.randint(0,Tmax)
    #generate private key
    private_key=DTQW_RING(l,s,k,t)
    print(f"message sent by Bob: {message}")
    crypted=encrypt(message, private_key)
    #receive message
    msg=decrypt(crypted,l,s,k,t)
    print(f"decrypted message: {msg}")
    return msg


#generic unitary ensemble
def Coin_operator(k):
    theta=2*np.pi/d
    U_k=np.zeros((2,2), dtype=np.complex128)
    U_k[0,0]=np.exp(1j*theta)*np.cos(theta)
    U_k[0,1]=np.exp(1j*theta)*np.sin(theta)
    U_k[1,0]=-np.exp(-1j*theta)*np.sin(theta)
    U_k[1,1]=np.exp(-1j*theta)*np.cos(theta)
    return U_k

def Translation_operator(m):
        T=np.zeros((N,N))
        for j in range(N):
            T[(j+m)%N, j]=1
        return T

def DTQW_RING(l,s,k,t):
    #coin states
    heads=np.zeros(2)
    tails=np.zeros(2)
    heads[0]=1
    tails[1]=1

    #shift operator
    S=np.kron(Translation_operator(1), np.outer(heads, heads))+np.kron(Translation_operator(-1), np.outer(tails, tails))
    #coint operator
    Uc=Coin_operator(k)

    #initial state
    psi0=np.zeros((N,2))
    psi0[l,s]=1

    #time evolution
    psi0_flat = psi0.reshape(2*N)
    private_key=np.linalg.matrix_power(S@(np.kron(np.identity(N), Uc)), t)@psi0_flat
    private_key=private_key.reshape(N,2)
    return private_key

def encrypt(message, private_key):
    temp = string_to_binary(message)
    m = binary_to_int(temp)
    
    pk_flat = private_key.reshape(2*N)
    T_m = np.kron(Translation_operator(m), np.identity(2))
    psi_m_flat = T_m @ pk_flat
    
    return psi_m_flat.reshape(N, 2)

def decrypt(msg,l,s,k,t):
        #coin states
    heads=np.zeros(2)
    tails=np.zeros(2)
    heads[0]=1
    tails[1]=1
     #shift operator
    S=np.kron(Translation_operator(1), np.outer(heads, heads.conj()))+np.kron(Translation_operator(-1), np.outer(tails, tails.conj()))
    #coint operator
    Uc=Coin_operator(k)
    #time evolution (backwards)
    U=S@(np.kron(np.identity(N), Uc))
    U_dagger=U.conj().T
    msg_flat = msg.reshape(2*N)
    psi_f=np.linalg.matrix_power(U_dagger, t)@msg_flat
    psi_f=psi_f.reshape(N,2)

    #measure
    probs = np.sum(np.abs(psi_f)**2, axis=1)
    m_prime = np.argmax(probs)
    m=(m_prime-l)%N

    #char
    temp=int_to_binary(m)
    message=binary_to_string(temp)
    return message

     
     


string='c'
send_message_to_Alice(string)


message sent by Bob: c
decrypted message: c


In [3]:
def send_full_message(message: str) -> str:
    # Encrypts and decrypts a multi-character string one letter at a time
    print(f"{'='*50}")
    print(f"Original message: '{message}'  ({len(message)} chars)")
    print(f"{'='*50}")

    decrypted = ""
    for i, char in enumerate(message):
        print(f"\nStep {i+1}/{len(message)}:")
        decrypted += send_message_to_Alice(char)

    print(f"\n{'='*50}")
    print(f"Recovered message: '{decrypted}'")
    print(f"Success: {decrypted == message}")
    print(f"{'='*50}")
    return decrypted


send_full_message("top secret")

Original message: 'top secret'  (10 chars)

Step 1/10:
message sent by Bob: t
decrypted message: t


TypeError: can only concatenate str (not "NoneType") to str